# NB07: Global LightGBM Revenue Forecast Model

Direct revenue forecasting with a separate LightGBM model.
Loads revenue-specific features from NB06 (`revenue_modeling_dataset.csv`).

**Pipeline:** NB03 (shared features) → NB06 (revenue features) → NB07 (model)

**Key changes from v1:**
- Loads from NB06 output (not NB03 directly)
- Dynamic feature detection from NB06 manifest (no hardcoded lists)
- Recursive forward forecast (predictions feed back as lags)
- Intermittency features moved to NB06

In [43]:
import sys
import os

_this_dir = os.path.dirname(os.path.abspath('__file__'))
_candidates = [
    os.environ.get('UC4_PROJECT_ROOT', ''),
    os.path.join(_this_dir, '..'),
]
for _c in _candidates:
    _test = os.path.join(_c, 'data', 'processed', 'modeling_dataset.csv')
    if os.path.exists(_test):
        _project_root = _c
        break
else:
    raise FileNotFoundError("Cannot find project root. Set UC4_PROJECT_ROOT or run from notebooks/")

_pylibs = os.path.join(os.path.dirname(_project_root), '.pylibs')
if os.path.isdir(_pylibs):
    sys.path.insert(0, _pylibs)

import pandas as pd
import numpy as np
import lightgbm as lgb
import json as _json
from sklearn.metrics import r2_score, mean_absolute_error
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

project_root = Path(_project_root)
data_dir = project_root / 'data' / 'processed'
fig_dir = project_root / 'reports' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

# ========== 1. LOAD NB06 OUTPUT ==========
print("=" * 60)
print("NB07: GLOBAL LIGHTGBM REVENUE FORECAST MODEL")
print("=" * 60)

# Load from NB06 (revenue_modeling_dataset.csv)
# Falls back to NB03 (modeling_dataset.csv) if NB06 hasn't been run
nb06_path = data_dir / 'revenue_modeling_dataset.csv'
nb03_path = data_dir / 'modeling_dataset.csv'

if nb06_path.exists():
    df = pd.read_csv(nb06_path)
    print(f"\nLoaded NB06 output: {nb06_path.name}")
else:
    df = pd.read_csv(nb03_path)
    print(f"\nWARNING: NB06 output not found — loading NB03 directly")
    print(f"  Run NB06 first for revenue-specific features")

df['week_ending'] = pd.to_datetime(df['week_ending'])

print(f"Data: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Stores: {df['store_code'].nunique()}, Divisions: {df['division_code'].nunique()}")
print(f"Groups: {df.groupby(['store_code','division_code']).ngroups}")
print(f"Date range: {df['week_ending'].min().date()} to {df['week_ending'].max().date()}")

# ========== DYNAMIC FEATURE DETECTION ==========
TARGET = 'revenue'

# Read manifest if available, otherwise detect from columns
manifest_path = data_dir / 'nb06_feature_manifest.json'
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = _json.load(f)
    print(f"\nLoaded feature manifest (NB06 added {len(manifest.get('nb06_added_columns', []))} features)")

# Exclude non-feature columns
EXCLUDE_COLS = {'week_ending', 'store_code', 'city', 'division_code', 'province',
                'units', 'revenue', 'revenue_clean', 'is_return',
                'store_enc', 'div_enc', 'store_weekly_revenue'}

# All numeric columns that aren't excluded = features
FEATURES = sorted([
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in EXCLUDE_COLS
])

# Categorize for reporting (not for filtering — all are used)
LAG_FEATS = [f for f in FEATURES if 'lag' in f or 'roll' in f or 'volatility' in f]
WEATHER_FEATS = [f for f in FEATURES if any(w in f for w in ['temp', 'precip', 'rain', 'snow', 'sunshine', 'bad_w', 'cooling', 'heating'])]
SEASON_FEATS = [f for f in FEATURES if f.startswith(('sin_', 'cos_'))]
CALENDAR_FEATS = [f for f in FEATURES if f in ['week_of_yr', 'month', 'year_idx', 'quarter',
                  'is_summer_peak', 'is_spring_opening', 'is_fall_closing', 'is_winter_off',
                  'n_holidays', 'has_holiday']]
OTHER_FEATS = [f for f in FEATURES if f not in LAG_FEATS + WEATHER_FEATS + SEASON_FEATS + CALENDAR_FEATS]

print(f"\nFeatures detected: {len(FEATURES)} total")
print(f"  Lag/Rolling: {len(LAG_FEATS)}")
print(f"  Weather:     {len(WEATHER_FEATS)}")
print(f"  Seasonality: {len(SEASON_FEATS)}")
print(f"  Calendar:    {len(CALENDAR_FEATS)}")
print(f"  Other:       {len(OTHER_FEATS)}")

NB07: GLOBAL LIGHTGBM REVENUE FORECAST MODEL

Loaded NB06 output: revenue_modeling_dataset.csv
Data: 24,323 rows x 100 columns
Stores: 27, Divisions: 11
Groups: 287
Date range: 2023-11-05 to 2026-03-22

Loaded feature manifest (NB06 added 19 features)

Features detected: 88 total
  Lag/Rolling: 29
  Weather:     34
  Seasonality: 6
  Calendar:    10
  Other:       11


In [44]:
# ========== ENCODE CATEGORICALS ==========
from sklearn.preprocessing import LabelEncoder

le_store = LabelEncoder()
le_div = LabelEncoder()
df['store_enc'] = le_store.fit_transform(df['store_code'])
df['div_enc'] = le_div.fit_transform(df['division_code'])
CAT_ENC = ['store_enc', 'div_enc']

# No inline feature engineering — all features come from NB06
print(f"Categorical encoders fitted: {len(CAT_ENC)}")
print(f"Total features: {len(FEATURES)} numeric + {len(CAT_ENC)} categorical = {len(FEATURES) + len(CAT_ENC)}")

Categorical encoders fitted: 2
Total features: 88 numeric + 2 categorical = 90


In [45]:
# ========== DROP NaN WARMUP ==========
FEATURE_COLS = FEATURES + CAT_ENC

# Drop rows where critical lags are NaN (early weeks in each series)
warmup_cols = [c for c in ['units_lag_4w', 'avg_temp_lag_2w'] if c in df.columns]
if warmup_cols:
    df_model = df.dropna(subset=warmup_cols).copy()
else:
    df_model = df.copy()

print(f"\nAfter lag warmup drop: {len(df_model):,} rows ({df_model.groupby(['store_code','division_code']).ngroups} groups)")
print(f"Feature columns: {len(FEATURE_COLS)}")


After lag warmup drop: 22,588 rows (284 groups)
Feature columns: 90


## 2. Walk-Forward Train/Test Split

In [46]:
# ========== 2. WALK-FORWARD SPLIT ==========
df_model = df_model.sort_values(['week_ending', 'store_code', 'division_code']).reset_index(drop=True)
all_weeks = sorted(df_model['week_ending'].unique())
n_weeks = len(all_weeks)
holdout_weeks = 26
cutoff_idx = n_weeks - holdout_weeks
cutoff_date = all_weeks[cutoff_idx]

train_mask = df_model['week_ending'] < cutoff_date
test_mask = df_model['week_ending'] >= cutoff_date

X_train = df_model.loc[train_mask, FEATURE_COLS]
y_train = df_model.loc[train_mask, TARGET]
X_test = df_model.loc[test_mask, FEATURE_COLS]
y_test = df_model.loc[test_mask, TARGET]

print(f"\nTrain: {len(X_train):,} rows | {df_model.loc[train_mask, 'week_ending'].min().date()} to {df_model.loc[train_mask, 'week_ending'].max().date()}")
print(f"Test:  {len(X_test):,} rows | {df_model.loc[test_mask, 'week_ending'].min().date()} to {df_model.loc[test_mask, 'week_ending'].max().date()}")
print(f"Cutoff: {cutoff_date.date()}")
print(f"Features: {len(FEATURE_COLS)}")


Train: 19,012 rows | 2023-12-03 to 2025-09-21
Test:  3,576 rows | 2025-09-28 to 2026-03-22
Cutoff: 2025-09-28
Features: 90


## 3. Direct Revenue LightGBM Point Forecast

In [47]:
# ========== 3. DIRECT REVENUE LIGHTGBM POINT FORECAST ==========
print("\n" + "=" * 60)
print("TRAINING LIGHTGBM (Revenue Point Forecast)")
print("=" * 60)

lgb_params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'n_estimators': 500,
    'max_depth': 6,
    'num_leaves': 31,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 1.0,
    'min_child_samples': 20,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1,
}

model_rev = lgb.LGBMRegressor(**lgb_params)
model_rev.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(100)]
)

y_pred_rev = model_rev.predict(X_test)

print(f"\nTest MAE:  ${mean_absolute_error(y_test, y_pred_rev):,.0f}")
print(f"Test R²:   {r2_score(y_test, y_pred_rev):.4f}")


TRAINING LIGHTGBM (Revenue Point Forecast)
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 1149.81
[200]	valid_0's l1: 1041.17
Early stopping, best iteration is:
[194]	valid_0's l1: 1040.56

Test MAE:  $1,041
Test R²:   0.7963


## 4. Quantile Regression — Revenue Confidence Intervals

In [48]:
# ========== 4. QUANTILE REGRESSION (CIs) ==========
print("\n" + "=" * 60)
print("TRAINING QUANTILE MODELS (P5, P95)")
print("=" * 60)

model_rev_q05 = lgb.LGBMRegressor(**{**lgb_params, 'objective': 'quantile', 'alpha': 0.05, 'metric': 'quantile'})
model_rev_q95 = lgb.LGBMRegressor(**{**lgb_params, 'objective': 'quantile', 'alpha': 0.95, 'metric': 'quantile'})

model_rev_q05.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
model_rev_q95.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])

y_rev_lower = model_rev_q05.predict(X_test)
y_rev_upper = model_rev_q95.predict(X_test)
y_rev_lower = np.minimum(y_rev_lower, y_pred_rev)
y_rev_upper = np.maximum(y_rev_upper, y_pred_rev)

rev_coverage = np.mean((y_test >= y_rev_lower) & (y_test <= y_rev_upper))
avg_width = np.mean(y_rev_upper - y_rev_lower)
print(f"\n90% Prediction Interval Coverage: {rev_coverage:.1%} (target: 90%)")
print(f"Average interval width: ${avg_width:,.0f}")


TRAINING QUANTILE MODELS (P5, P95)

90% Prediction Interval Coverage: 82.9% (target: 90%)
Average interval width: $3,533


In [49]:
# ========== BUILD TEST DATAFRAME ==========
def wmape(actual, predicted):
    denom = np.sum(np.abs(actual))
    return np.sum(np.abs(actual - predicted)) / denom if denom > 0 else np.nan

test_df = df_model.loc[test_mask].copy()
test_df['pred_revenue'] = y_pred_rev
test_df['pred_revenue_lower'] = y_rev_lower
test_df['pred_revenue_upper'] = y_rev_upper
test_df['actual_revenue'] = test_df['revenue']

overall_rev_wmape = wmape(test_df['actual_revenue'].values, test_df['pred_revenue'].values)
print(f"\nOverall Revenue wMAPE: {overall_rev_wmape:.4f}")
print(f"Actual total:    ${test_df['actual_revenue'].sum():,.0f}")
print(f"Predicted total: ${test_df['pred_revenue'].sum():,.0f}")
pct_diff = (test_df['pred_revenue'].sum() / test_df['actual_revenue'].sum() - 1) * 100
print(f"Difference:      {pct_diff:+.1f}%")


Overall Revenue wMAPE: 0.3318
Actual total:    $10,969,115
Predicted total: $10,983,556
Difference:      +0.1%


## 5. Performance by Division & Store

In [50]:
# ========== 5. PERFORMANCE BY DIVISION ==========
print("\n" + "=" * 60)
print("PERFORMANCE BY DIVISION")
print("=" * 60)

div_metrics = []
for div in sorted(test_df['division_code'].unique()):
    sub = test_df[test_df['division_code'] == div]
    if sub['actual_revenue'].sum() == 0:
        continue
    div_metrics.append({
        'division': div,
        'n_groups': sub.groupby('store_code').ngroups,
        'n_rows': len(sub),
        'total_actual_revenue': round(sub['actual_revenue'].sum()),
        'revenue_wmape': round(wmape(sub['actual_revenue'].values, sub['pred_revenue'].values), 3),
        'revenue_r2': round(r2_score(sub['actual_revenue'].values, sub['pred_revenue'].values), 3),
    })

div_df = pd.DataFrame(div_metrics).sort_values('total_actual_revenue', ascending=False)
print(div_df.to_string(index=False))

print(f"\nOverall revenue wMAPE: {overall_rev_wmape:.3f}")


PERFORMANCE BY DIVISION
division  n_groups  n_rows  total_actual_revenue  revenue_wmape  revenue_r2
      SP        26     230               4382222          0.267       0.741
      PC        26     666               2513769          0.218       0.892
      ME        26     531               1370644          0.504       0.690
      FI        25     459               1018234          0.306       0.809
      GA        26     248                771559          0.473       0.528
      BQ        26     341                261148          0.502       0.541
      CH        24     177                197716          0.413       0.706
      HT        17      49                175315          0.312       0.923
      TO        25     129                103407          0.595       0.828
      PA        24     137                100184          0.766       0.505
      LO        26     609                 74917          1.744      -2.485

Overall revenue wMAPE: 0.332


In [51]:
# ========== 6. PERFORMANCE BY STORE ==========
print("\n" + "=" * 60)
print("PERFORMANCE BY STORE (Top 15)")
print("=" * 60)

store_metrics = []
for store in sorted(test_df['store_code'].unique()):
    sub = test_df[test_df['store_code'] == store]
    if sub['actual_revenue'].sum() == 0:
        continue
    store_metrics.append({
        'store': store,
        'n_divs': sub['division_code'].nunique(),
        'revenue_wmape': round(wmape(sub['actual_revenue'].values, sub['pred_revenue'].values), 3),
        'total_revenue': round(sub['actual_revenue'].sum()),
    })

store_df = pd.DataFrame(store_metrics).sort_values('total_revenue', ascending=False)
print(store_df.head(15).to_string(index=False))

worst = store_df[store_df['revenue_wmape'] > 0.5]
if len(worst) > 0:
    print(f"\nWarning: {len(worst)} stores with revenue wMAPE > 50%:")
    print(worst[['store', 'revenue_wmape', 'total_revenue']].to_string(index=False))


PERFORMANCE BY STORE (Top 15)
store  n_divs  revenue_wmape  total_revenue
 CP05      11          0.343        1372247
 CP07      10          0.306        1000028
 CP36      11          0.567         837024
 CP06      11          0.366         796390
 CP37      11          0.273         722273
 CP10      11          0.317         632650
CP202      10          0.169         445545
 CP17      10          0.250         423008
 CP35      11          0.196         419907
 CP47      11          0.411         406540
 CP04      10          0.308         391069
 CP13      11          0.248         377238
 CP02      10          0.185         358578
 CP16      11          0.368         353481
 CP46       9          0.180         292134

store  revenue_wmape  total_revenue
 CP36          0.567         837024
 CP12          0.594         278320
 CP15          0.509         156170


## 6. Feature Importance

In [52]:
# ========== 7. FEATURE IMPORTANCE ==========
print("\n" + "=" * 60)
print("FEATURE IMPORTANCE (Top 15)")
print("=" * 60)

importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model_rev.feature_importances_
}).sort_values('importance', ascending=False)

print(importance.head(15).to_string(index=False))

# Feature group contribution (using dynamic categories from Cell 1)
groups_map = {
    'Lag/Rolling': LAG_FEATS,
    'Weather': WEATHER_FEATS,
    'Seasonality': SEASON_FEATS,
    'Calendar': CALENDAR_FEATS,
    'Identity': CAT_ENC,
    'Other': OTHER_FEATS,
}
print("\nFeature Group Contribution:")
total_imp = importance['importance'].sum()
for name, feats in groups_map.items():
    present = [f for f in feats if f in importance['feature'].values]
    grp_total = importance.loc[importance['feature'].isin(present), 'importance'].sum()
    pct = grp_total / total_imp * 100
    print(f"  {name:25s}: {pct:5.1f}%")



FEATURE IMPORTANCE (Top 15)
             feature  importance
      n_transactions        1181
  avg_price_per_unit         718
  revenue_roll4_mean         362
      revenue_lag_1w         239
      price_momentum         204
 price_vs_div_median         180
    units_roll4_mean         130
        units_lag_1w         103
store_revenue_lag_1w         101
      revenue_lag_2w          99
        price_lag_1w          96
    price_roll4_mean          96
          sin_week_1          93
    weeks_with_sales          93
      revenue_lag_4w          84

Feature Group Contribution:
  Lag/Rolling              :  41.3%
  Weather                  :  11.6%
  Seasonality              :   3.0%
  Calendar                 :   0.6%
  Identity                 :   1.7%
  Other                    :  43.4%


## 7. Walk-Forward Cross-Validation

In [53]:
# ========== 8. WALK-FORWARD CV ==========
print("\n" + "=" * 60)
print("WALK-FORWARD CROSS-VALIDATION (6 folds)")
print("=" * 60)

HORIZON = 4
N_FOLDS = 6
MIN_TRAIN_WEEKS = 52
all_weeks_sorted = sorted(df_model['week_ending'].unique())
total_weeks = len(all_weeks_sorted)
fold_starts = np.linspace(MIN_TRAIN_WEEKS, total_weeks - HORIZON, N_FOLDS + 1, dtype=int)[:-1]

cv_results = []
for fold_i, start_idx in enumerate(fold_starts):
    test_start = all_weeks_sorted[start_idx]
    test_end = all_weeks_sorted[min(start_idx + HORIZON - 1, total_weeks - 1)]
    
    tr = df_model['week_ending'] < test_start
    te = (df_model['week_ending'] >= test_start) & (df_model['week_ending'] <= test_end)
    if tr.sum() == 0 or te.sum() == 0:
        continue
    
    fold_model = lgb.LGBMRegressor(**{**lgb_params, 'n_estimators': 300, 'verbose': -1})
    fold_model.fit(df_model.loc[tr, FEATURE_COLS], df_model.loc[tr, TARGET])
    y_hat = fold_model.predict(df_model.loc[te, FEATURE_COLS])
    y_true = df_model.loc[te, TARGET].values
    
    fold_mae = mean_absolute_error(y_true, y_hat)
    fold_wmape = wmape(y_true, y_hat)
    fold_r2 = r2_score(y_true, y_hat)
    
    m = test_start.month
    season = 'Summer' if m in [6,7,8] else 'Winter' if m in [12,1,2] else 'Spring' if m in [3,4,5] else 'Fall'
    
    cv_results.append({
        'fold': fold_i + 1, 'train_rows': tr.sum(), 'test_rows': te.sum(),
        'test_period': f"{test_start.date()} to {test_end.date()}", 'season': season,
        'mae': fold_mae, 'wmape': fold_wmape, 'r2': fold_r2,
    })
    print(f"Fold {fold_i+1}: {test_start.date()} to {test_end.date()} ({season}) | MAE=${fold_mae:,.0f}, wMAPE={fold_wmape:.3f}, R²={fold_r2:.3f}")

cv_df = pd.DataFrame(cv_results)
print(f"\nCV Mean wMAPE: {cv_df['wmape'].mean():.3f} ± {cv_df['wmape'].std():.3f}")
print(f"CV Mean MAE:   ${cv_df['mae'].mean():,.0f} ± ${cv_df['mae'].std():,.0f}")
print(f"CV Mean R²:    {cv_df['r2'].mean():.3f} ± {cv_df['r2'].std():.3f}")


WALK-FORWARD CROSS-VALIDATION (6 folds)
Fold 1: 2024-12-01 to 2024-12-22 (Winter) | MAE=$1,201, wMAPE=0.325, R²=0.632
Fold 2: 2025-02-09 to 2025-03-02 (Winter) | MAE=$1,057, wMAPE=0.251, R²=0.921
Fold 3: 2025-04-27 to 2025-05-18 (Spring) | MAE=$3,945, wMAPE=0.183, R²=0.918
Fold 4: 2025-07-13 to 2025-08-03 (Summer) | MAE=$2,857, wMAPE=0.203, R²=0.886
Fold 5: 2025-09-28 to 2025-10-19 (Fall) | MAE=$1,619, wMAPE=0.268, R²=0.861
Fold 6: 2025-12-14 to 2026-01-04 (Winter) | MAE=$778, wMAPE=0.420, R²=0.627

CV Mean wMAPE: 0.275 ± 0.087
CV Mean MAE:   $1,909 ± $1,236
CV Mean R²:    0.808 ± 0.140


## 8. Baseline Comparison

In [54]:
# ========== 9. BASELINE COMPARISON ==========
print("\n" + "=" * 60)
print("BASELINE COMPARISON")
print("=" * 60)

test_groups = df_model.loc[test_mask].copy()
test_groups['lgbm_pred'] = y_pred_rev

test_groups['naive_seasonal'] = test_groups['revenue_lag_4w']
test_groups['naive_ma4'] = test_groups['revenue_roll4_mean']
div_means = df_model.loc[train_mask].groupby('division_code')['revenue'].mean()
test_groups['naive_div_mean'] = test_groups['division_code'].map(div_means)

baselines = {
    'Seasonal Naive (4w)': 'naive_seasonal',
    'Moving Average (4w)': 'naive_ma4',
    'Division Mean': 'naive_div_mean',
    'Global LightGBM': 'lgbm_pred'
}

print(f"{'Model':<30s} {'MAE':>10s} {'wMAPE':>8s} {'R²':>8s}")
print("-" * 58)
for name, col in baselines.items():
    valid = test_groups[[TARGET, col]].dropna()
    if len(valid) == 0:
        continue
    mae = mean_absolute_error(valid[TARGET], valid[col])
    w = wmape(valid[TARGET].values, valid[col].values)
    r2 = r2_score(valid[TARGET], valid[col])
    marker = " <- OURS" if col == 'lgbm_pred' else ""
    print(f"{name:<30s} ${mae:>9,.0f} {w:>8.3f} {r2:>8.3f}{marker}")


BASELINE COMPARISON
Model                                 MAE    wMAPE       R²
----------------------------------------------------------
Seasonal Naive (4w)            $    3,453    1.101   -0.361
Moving Average (4w)            $    2,537    0.809    0.289
Division Mean                  $    8,875    2.830   -1.006
Global LightGBM                $    1,041    0.332    0.796 <- OURS


## 9. 4-Week Forward Forecast

In [55]:
# ========== 10. GENERATE 4-WEEK RECURSIVE FORECAST ==========
print("\n" + "=" * 60)
print("4-WEEK FORWARD FORECAST (Revenue, Recursive)")
print("=" * 60)

latest_week = df_model['week_ending'].max()
current_state = df_model.sort_values('week_ending').groupby(['store_code', 'division_code']).last().reset_index()

# Revenue lag columns to update recursively
REV_LAG_MAP = {
    'revenue_lag_1w': 1, 'revenue_lag_2w': 2,
    'revenue_lag_4w': 4, 'revenue_lag_8w': 8, 'revenue_lag_12w': 12,
}
REV_ROLL_COLS = ['revenue_roll4_mean', 'revenue_roll8_mean', 'revenue_roll12_mean']

forecast_rows = []
# Keep a history buffer per group for rolling computation
group_histories = {}
for (s, d), grp in df_model.groupby(['store_code', 'division_code']):
    group_histories[(s, d)] = list(grp.sort_values('week_ending')['revenue'].tail(12).values)

for horizon in range(1, 5):
    fwd = current_state.copy()
    fwd['forecast_week'] = latest_week + pd.Timedelta(weeks=horizon)
    fwd['horizon'] = horizon

    if horizon > 1:
        # Update revenue lag features using previous predictions
        prev_preds = forecast_rows[-1]  # last horizon's predictions
        pred_map = prev_preds.set_index(['store_code', 'division_code'])['pred_revenue'].to_dict()

        for idx, row in fwd.iterrows():
            key = (row['store_code'], row['division_code'])
            prev_pred = pred_map.get(key, 0)

            # Shift lags: lag_1w = last prediction, lag_2w = old lag_1w, etc.
            if 'revenue_lag_2w' in fwd.columns:
                fwd.at[idx, 'revenue_lag_2w'] = fwd.at[idx, 'revenue_lag_1w']
            if 'revenue_lag_1w' in fwd.columns:
                fwd.at[idx, 'revenue_lag_1w'] = prev_pred

            # Update rolling means from history buffer
            if key in group_histories:
                hist = group_histories[key]
                hist.append(prev_pred)
                if 'revenue_roll4_mean' in fwd.columns:
                    fwd.at[idx, 'revenue_roll4_mean'] = np.mean(hist[-4:])
                if 'revenue_roll8_mean' in fwd.columns and len(hist) >= 8:
                    fwd.at[idx, 'revenue_roll8_mean'] = np.mean(hist[-8:])
                if 'revenue_roll12_mean' in fwd.columns and len(hist) >= 12:
                    fwd.at[idx, 'revenue_roll12_mean'] = np.mean(hist[-12:])

    # Ensure only valid feature columns
    X_fwd = fwd[FEATURE_COLS].fillna(0)

    fwd['pred_revenue'] = model_rev.predict(X_fwd)
    fwd['pred_revenue_lower'] = model_rev_q05.predict(X_fwd)
    fwd['pred_revenue_upper'] = model_rev_q95.predict(X_fwd)

    out = fwd[['store_code', 'division_code', 'forecast_week', 'horizon',
                'pred_revenue', 'pred_revenue_lower', 'pred_revenue_upper']].copy()
    forecast_rows.append(out)

forecast_df = pd.concat(forecast_rows, ignore_index=True)
print(f"Forecast: {forecast_df.shape[0]} rows ({forecast_df['store_code'].nunique()} stores x {forecast_df['division_code'].nunique()} divisions x 4 weeks)")
print(f"\nTotal predicted revenue (4 weeks): ${forecast_df['pred_revenue'].sum():,.0f}")
print(f"  Revenue range: ${forecast_df['pred_revenue_lower'].sum():,.0f} to ${forecast_df['pred_revenue_upper'].sum():,.0f}")

# Compare week 1 vs week 4 to see decay from recursive lag degradation
w1 = forecast_df[forecast_df['horizon']==1]['pred_revenue'].sum()
w4 = forecast_df[forecast_df['horizon']==4]['pred_revenue'].sum()
print(f"\nWeek 1 total: ${w1:,.0f}")
print(f"Week 4 total: ${w4:,.0f}  ({(w4/w1-1)*100:+.1f}% vs week 1)")


4-WEEK FORWARD FORECAST (Revenue, Recursive)
Forecast: 1136 rows (26 stores x 11 divisions x 4 weeks)

Total predicted revenue (4 weeks): $2,213,453
  Revenue range: $1,350,028 to $4,187,896

Week 1 total: $567,968
Week 4 total: $554,871  (-2.3% vs week 1)


## 10. Save Outputs

In [56]:
# ========== 11. SAVE OUTPUTS ==========
print("\n" + "=" * 60)
print("SAVING OUTPUTS")
print("=" * 60)

forecast_df.to_csv(data_dir / 'nb07_forecast_output.csv', index=False)
print(f"ok nb07_forecast_output.csv")

group_accuracy = []
for (s, d), g in test_df.groupby(['store_code', 'division_code']):
    group_accuracy.append({
        'store_code': s, 'division_code': d,
        'actual_revenue': g['actual_revenue'].sum(), 'pred_revenue': g['pred_revenue'].sum(),
        'revenue_wmape': wmape(g['actual_revenue'].values, g['pred_revenue'].values),
        'n_weeks': len(g),
    })
pd.DataFrame(group_accuracy).to_csv(data_dir / 'nb07_model_accuracy.csv', index=False)
print(f"ok nb07_model_accuracy.csv")

div_df.to_csv(data_dir / 'nb07_division_accuracy.csv', index=False)
print(f"ok nb07_division_accuracy.csv")

cv_df.to_csv(data_dir / 'nb07_cv_results.csv', index=False)
print(f"ok nb07_cv_results.csv")

importance.to_csv(data_dir / 'nb07_feature_importance.csv', index=False)
print(f"ok nb07_feature_importance.csv")

summary = {
    'model': 'Global LightGBM (direct revenue)',
    'train_rows': len(X_train), 'test_rows': len(X_test),
    'n_features': len(FEATURE_COLS),
    'n_groups': df_model.groupby(['store_code', 'division_code']).ngroups,
    'n_stores': df_model['store_code'].nunique(),
    'n_divisions': df_model['division_code'].nunique(),
    'test_revenue_wmape': overall_rev_wmape,
    'test_revenue_r2': r2_score(y_test, y_pred_rev),
    'cv_wmape_mean': cv_df['wmape'].mean(),
    'cv_wmape_std': cv_df['wmape'].std(),
    'revenue_pi_coverage_90': rev_coverage,
    'total_forecast_revenue_4w': forecast_df['pred_revenue'].sum(),
}
pd.DataFrame([summary]).to_csv(data_dir / 'nb07_summary.csv', index=False)
print(f"ok nb07_summary.csv")

# ========== FINAL SUMMARY ==========
print("\n" + "=" * 60)
print("NB07 COMPLETE - FINAL SUMMARY")
print("=" * 60)
print(f"Model:         Global LightGBM (direct revenue)")
print(f"Groups:        {df_model.groupby(['store_code','division_code']).ngroups} (100% coverage)")
print(f"Train/Test:    {len(X_train):,} / {len(X_test):,}")
print(f"Features:      {len(FEATURE_COLS)}")
print(f"Revenue wMAPE: {overall_rev_wmape:.3f} (R²={r2_score(y_test, y_pred_rev):.4f})")
print(f"CV wMAPE:      {cv_df['wmape'].mean():.3f} +/- {cv_df['wmape'].std():.3f}")
print(f"90% CI:        {rev_coverage:.1%}")
print(f"4wk Forecast:  ${forecast_df['pred_revenue'].sum():,.0f}")


SAVING OUTPUTS
ok nb07_forecast_output.csv
ok nb07_model_accuracy.csv
ok nb07_division_accuracy.csv
ok nb07_cv_results.csv
ok nb07_feature_importance.csv
ok nb07_summary.csv

NB07 COMPLETE - FINAL SUMMARY
Model:         Global LightGBM (direct revenue)
Groups:        284 (100% coverage)
Train/Test:    19,012 / 3,576
Features:      90
Revenue wMAPE: 0.332 (R²=0.7963)
CV wMAPE:      0.275 +/- 0.087
90% CI:        82.9%
4wk Forecast:  $2,213,453
